In [6]:
import os
import sys
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
class TextDataset:
    def __init__(self, file_path, tokenize=True):
        # 读取文本并按需进行分词
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()
        if tokenize:
            self.sentences = sent_tokenize(text.lower())
            self.tokens = [word_tokenize(sent) for sent in self.sentences]
        else:
            self.text = text

    def build_vocab(self, min_freq=1):
        # 统计词频
        frequency = defaultdict(int)
        for sentence in self.tokens:
            for token in sentence:
                frequency[token] += 1
        self.frequency = frequency

        # 加入<unk>处理未登录词，加入<pad>用于对齐变长输入进而加速
        self.token2id = {'<unk>': 1, '<pad>': 0}
        self.id2token = {1: '<unk>', 0: '<pad>'}
        for token, freq in sorted(frequency.items(), key=lambda x: -x[1]):
            # 丢弃低频词
            if freq > min_freq:
                self.token2id[token] = len(self.token2id)
                self.id2token[len(self.id2token)] = token
            else:
                break

    def get_word_distribution(self):
        distribution = np.zeros(len(self.token2id))
        for token, freq in self.frequency.items():
            token_id = self.token2id.get(token, self.token2id['<unk>'])
            distribution[token_id] += freq
        distribution /= distribution.sum()
        return distribution

    # 将分词结果转化为索引表示
    def convert_tokens_to_ids(self, drop_single_word=True):
        self.token_ids = []
        for sentence in self.tokens:
            token_ids = [self.token2id.get(token, 1) for token in sentence]
            # 忽略只有一个token的序列，无法计算loss
            if len(token_ids) == 1 and drop_single_word:
                continue
            self.token_ids.append(token_ids)
        return self.token_ids


In [3]:
from collections import defaultdict
import re
import numpy as np

try:
    from nltk import sent_tokenize as _nltk_sent_tokenize
    from nltk import word_tokenize as _nltk_word_tokenize

    # 主动测试资源是否存在，避免在 TextDataset 初始化时才报错
    _nltk_sent_tokenize("test sentence.")
    _nltk_word_tokenize("test sentence.")
    sent_tokenize = _nltk_sent_tokenize
    word_tokenize = _nltk_word_tokenize
    print("使用 NLTK 分词")
except LookupError:
    print("使用本地正则分词")

    def sent_tokenize(text):
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

    def word_tokenize(sentence):
        return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\w\s]", sentence)


dataset = TextDataset(file_path=r"D:\DeepLearning\DataSet\Text\anna.txt")


使用本地正则分词


接下来建立词表，截断过长的序列，将序列填充（padding）到相同长度（不使用词嵌入）

In [4]:
import numpy as np

dataset.build_vocab()
sent_tokens = dataset.convert_tokens_to_ids()
# 截断和填充
max_len=40
for i, tokens in enumerate(sent_tokens):
    tokens = tokens[:max_len]
    tokens += [dataset.token2id['<pad>']] * (max_len - len(tokens))
    sent_tokens[i] = tokens
    
sent_tokens = np.array(sent_tokens)

下面来实现Transformer模型，包括加入了位置编码的嵌入层、缩放点乘注意力、多头注意力、层归一化等具体实现。

In [7]:
# 实现Transformer模型
class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, max_len, embed_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.embed_size = embed_size
        self.word_embedding = nn.Embedding(vocab_size, embed_size)
        self.pos_embedding = nn.Embedding(max_len, embed_size)
        
    def forward(self, input_ids, pos_ids):
        """
        input_ids/pos_ids: batch_size * seq_len
        return: batch_size * seq_len * embed_size
        """
        word_embed = self.word_embedding(input_ids)
        pos_embed = self.pos_embedding(pos_ids)
        # 将词嵌入和位置嵌入相加得到嵌入层输出
        return word_embed + pos_embed

# 缩放点乘注意力
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, queries, keys, values, attention_mask):
        """
        queries/keys/values: batch_size * seq_len * hidden_size
        attention_mask: batch_size * seq_len * seq_len
        return: batch_size * seq_len * hidden_size
        """
        d = queries.size(-1)
        # 根据点乘注意力的矩阵形式计算注意力分数，除以查询向量或键向量
        # 维度的平方根，即为缩放点乘注意力
        scores = torch.bmm(queries, torch.transpose(keys, 1, 2)) / np.sqrt(d)
        # 将掩码为0的位置的注意力分数设为一个大负数，根据softmax函数
        # 的性质，这些注意力分数归一化后接近0
        scores[attention_mask == 0] = -1e6
        self.attention_weights = F.softmax(scores, dim=-1)
        return torch.bmm(self.dropout(self.attention_weights), values)
    
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)
        self.W_o = nn.Linear(hidden_size, hidden_size)
        self.attention = ScaledDotProductAttention(dropout)
    
    def transpose_qkv(self, states):
        # 将长度为hidden_size的向量分成num_heads个长度相等的向量
        states = states.reshape(states.shape[0], states.shape[1],\
            self.num_heads, self.hidden_size // self.num_heads)
        states = torch.permute(states, (0, 2, 1, 3))
        return states.reshape(-1, states.shape[2], states.shape[3])
    
    # 与transpose_qkv的变换相反
    def transpose_output(self, states):
        states = states.reshape(-1, self.num_heads, states.shape[1],\
            states.shape[2])
        states = torch.permute(states, (0, 2, 1, 3))
        return states.reshape(states.shape[0], states.shape[1], -1)
    
    def forward(self, queries, keys, values, attention_mask):
        """
        querys/keys/values: batch * seq_len * hidden_size
        attention_mask: batch * seq_len * seq_len
        return:
        """
        # (batch_size * num_heads) * seq_len * (hidden_size / num_heads)
        queries = self.transpose_qkv(self.W_q(queries))
        keys = self.transpose_qkv(self.W_k(keys))
        values = self.transpose_qkv(self.W_v(values))
        # 重复张量的元素，用以支持多个注意力头的运算
        # (batch_size * num_heads) * seq_len * seq_len
        attention_mask = torch.repeat_interleave(attention_mask,\
            repeats=self.num_heads, dim=0)
        # (batch_size * num_heads) * seq_len * (hidden_size / num_heads)
        output = self.attention(queries, keys, values, attention_mask)
        # batch * seq_len * hidden_size
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)

# 两层前馈神经网络
class PositionWiseFNN(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.dense1 = nn.Linear(hidden_size, intermediate_size)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(intermediate_size, hidden_size)
        
    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

# 层归一化
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))
        # 一个小量用于数值稳定（防止除0）
        self.eps = eps
        
    def forward(self, hidden_states):
        mean = torch.mean(hidden_states, -1, keepdim=True)
        std = torch.std(hidden_states, -1, keepdim=True)
        return self.gamma * (hidden_states - mean) / (std +\
            self.eps) + self.beta

# 将两个输入相加并归一化
class AddNorm(nn.Module):
    def __init__(self, hidden_size, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = LayerNorm(hidden_size)
        
    def forward(self, X, Y):
        return self.layer_norm(self.dropout(Y) + X)
    
# 一个完整的Transformer层
class TransformerLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout, intermediate_size):
        super().__init__()
        self.self_attention = MultiHeadSelfAttention(hidden_size,\
            num_heads, dropout)
        self.add_norm1 = AddNorm(hidden_size, dropout)
        self.fnn = PositionWiseFNN(hidden_size, intermediate_size)
        self.add_norm2 = AddNorm(hidden_size, dropout)
    
    def forward(self, X, attention_mask):
        Y = self.add_norm1(X, self.self_attention(X, X, X, attention_mask))
        return self.add_norm2(Y, self.fnn(Y))

In [8]:
# 在Transformer模型基础上加上语言模型需要的输入输出、损失计算等
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, max_len, hidden_size, num_layers,\
                 num_heads, dropout, intermediate_size):
        super().__init__()
        self.embedding_layer = EmbeddingLayer(vocab_size, max_len,\
            hidden_size)
        self.num_layers = num_layers
        # 使用ModuleList保存多个Transformer层，注意不能使用Python列表，
        # Python列表保存的PyTorch变量无法自动求导
        self.layers = nn.ModuleList([TransformerLayer(hidden_size,\
            num_heads, dropout, intermediate_size)\
            for _ in range(num_layers)])
        self.output_layer = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, input_ids):
        # 这里实现的forward()函数一次只能处理一句话，
        # 如果想要支持批次运算，实现起来会更复杂，也会引入冗余操作
        seq_len = input_ids.size(0)
        assert input_ids.ndim == 1 and seq_len <= \
            self.embedding_layer.max_len
        
        # 1 * seq_len
        input_ids = torch.unsqueeze(input_ids, dim=0)
        pos_ids = torch.unsqueeze(torch.arange(seq_len), dim=0)
        # 定义下三角掩码，用于语言模型训练
        # 1 * seq_len * seq_len
        attention_mask = torch.unsqueeze(torch.tril(torch.ones((seq_len,\
            seq_len), dtype=torch.int32)), dim=0)
        # 1 * seq_len * hidden_size
        hidden_states = self.embedding_layer(input_ids, pos_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask)
        outputs = self.output_layer(hidden_states)
        
        loss_fct = nn.CrossEntropyLoss(ignore_index=0)
        loss = loss_fct(outputs[:, :-1].squeeze(),\
            input_ids[:, 1:].squeeze())
        return loss

In [12]:
from torch.utils.data import DataLoader
from torch.optim import SGD, Adam
from tqdm import tqdm, trange

In [ ]:
# 梯度裁剪
def grad_clipping(model, theta=1):
    params = [p for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not params:
        return
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

# 训练TransformerLM，由于不再采取批次训练，因此不再使用RNNLM和data_loader
def train_transformer_lm(data, model, epochs=50, learning_rate=1e-3):
    optimizer = Adam(model.parameters(), lr=learning_rate)
    model.zero_grad()
    model.train()
    
    epoch_loss = []
    with trange(epochs, desc='epoch', ncols=60) as pbar:
        for epoch in pbar:
            step_loss = []
            np.random.shuffle(data)
            for step, x in enumerate(data):
                loss = model(torch.tensor(x, dtype=torch.long))
                pbar.set_description(f'epoch: {epoch},'+\
                    f' loss={loss.item():.4f}')
                loss.backward()
                grad_clipping(model)
                optimizer.step()
                model.zero_grad()
                step_loss.append(loss.item())
            # 本章前面的模型训练使用batch_size为16，
            # TransformerLM出于简便实现只能使用batch_size为1
            # 因此TransformerLM每一步的损失方差会更大，
            # 为便于对比，取每个epoch最后16个样本的平均损失
            epoch_loss.append(np.mean(step_loss[-16:]))
    
    epoch_loss = np.array(epoch_loss)
    plt.plot(range(len(epoch_loss)), epoch_loss)
    plt.xlabel('training epoch')
    plt.ylabel('loss')
    plt.show()
    


In [16]:
sent_tokens = dataset.convert_tokens_to_ids()
max_len=40
for i, tokens in enumerate(sent_tokens):
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))
    sent_tokens[i] = tokens
sent_tokens = np.array(sent_tokens)
vocab_size = len(dataset.token2id)

model = TransformerLM(vocab_size, max_len=40, hidden_size=128,\
    num_layers=1, num_heads=4, dropout=0., intermediate_size=512)
    
train_transformer_lm(sent_tokens, model)

epoch-0, loss=3.7238:   0%|          | 0/50 [00:42<?, ?it/s]


KeyboardInterrupt: 